# Fig S11a — AD GWAS SER fallback frequencies (data processing)

Share of study–region analyses in which SuSiE-rRSS used the SER fallback (`serFallbackUsed` in `export/SER_flag.tsv`, as in Fig 6c). Denominator: study–region analyses whose final SuSiE-rRSS result reports at least one credible set (the eb_mix export, `cs_coverage_0.95`; the same 338 analyses as Fig 6b,e). A region without a reported credible set needs no fallback, so it is not counted; flagged analyses whose SER model reports no credible set are listed separately (`n_flagged_no_cs`), and `n_analyzed` keeps the number of all analyzed regions.

The export is taken as the final pipeline result (SER model when flagged); cell 1 checks that no flagged analysis has more than one exported credible set, which the one-effect SER model cannot produce.

- **B3** all datasets (Results l.301–302)
- **B4** APOE region (`apoe_block`, Results l.337)
- **B5** per dataset (Fig S11a)
- **B6** distinct regions with multiple credible sets (Results l.322–323; eb_mix export)

**Output:** `figure_data/FigS12A.AD_GWAS_SER_Fallback_Frequency.{rds,csv}`

In [1]:
source("../../R/utils.R")

data_name <- "FigS12A.AD_GWAS_SER_Fallback_Frequency"
fm_root   <- "/hpc/users/sunh17/haochen/rss_qc/2026_output/ADGWAS_finemap/AD_GWAS_EUR_rss_L10/AD_GWAS_rss_ebmix_r4_L10"

## every analyzed study x region: one fine-mapping output per pair ("<study>.<block_id>.gwas_finemap.rds")
analyzed <- tibble(file = list.files(file.path(fm_root, "output/fine_mapping"), pattern = "\\.gwas_finemap\\.rds$")) %>%
  transmute(study    = str_split_fixed(file, "\\.", 3)[, 1],
            block_id = str_split_fixed(file, "\\.", 3)[, 2])

## SER fallback flag per study x region (same table as Fig 6c)
SER_flag_tbl <- read_tsv(file.path(fm_root, "export/SER_flag.tsv"), show_col_types = FALSE) %>%
  distinct(study, block_id, .keep_all = TRUE)

## check: the flag table covers the analyzed regions
tibble(n_analyzed          = nrow(analyzed),
       n_in_flag_table     = nrow(semi_join(analyzed, SER_flag_tbl, by = c("study", "block_id"))),
       n_flag_not_analyzed = nrow(anti_join(SER_flag_tbl, analyzed, by = c("study", "block_id"))))

## credible sets of the final SuSiE-rRSS result per study x region (eb_mix export, filtered as in Fig 6c)
eb_mix <- read_gwas_tables(file.path(fm_root, "export/")) %>%
  filter(!is.na(cs_coverage_0.95), !cs_coverage_0.95 %in% c("", "0", "susie_rss_0"))
cs_per_region <- eb_mix %>%
  distinct(study, block_id, cs = cs_coverage_0.95) %>%
  count(study, block_id, name = "n_cs")

flags <- analyzed %>%
  left_join(select(SER_flag_tbl, study, block_id, serFallbackUsed), by = c("study", "block_id")) %>%
  left_join(cs_per_region, by = c("study", "block_id")) %>%
  mutate(fallback = serFallbackUsed %in% TRUE,
         n_cs     = coalesce(n_cs, 0L),
         has_cs   = n_cs > 0)

## check: the export is the final result (a flagged analysis has at most the SER model's one credible set)
stopifnot("an exported region is not among the analyzed fits" = nrow(anti_join(cs_per_region, analyzed, by = c("study", "block_id"))) == 0,
          "a flagged analysis has > 1 exported credible set: the export is not the post-fallback result" = all(flags$n_cs[flags$fallback] <= 1))
flags %>% count(fallback, has_cs)

n_analyzed,n_in_flag_table,n_flag_not_analyzed
<int>,<int>,<int>
16296,338,0


fallback,has_cs,n
<lgl>,<lgl>,<int>
FALSE,FALSE,15958
FALSE,TRUE,314
TRUE,TRUE,24


In [2]:
## ---- B3: all datasets (Results l.301-302) ----
## denominator: analyses with a credible set in the final result; flagged analyses without one are listed apart
fallback_summary <- function(df) df %>%
  summarise(n_analyzed = n(), n_with_cs = sum(has_cs), n_fallback = sum(fallback & has_cs),
            n_flagged_no_cs = sum(fallback & !has_cs), .groups = "drop") %>%
  mutate(pct = round(100 * n_fallback / n_with_cs, 1))

overall <- fallback_summary(flags)
overall
cat(sprintf("Across all 12 datasets, SER fallback was used in %.1f%% (%d/%d) of study-region analyses with at least one credible set in the final SuSiE-rRSS result; a further %d flagged analyses reported no credible set after fallback.\n",
            overall$pct, overall$n_fallback, overall$n_with_cs, overall$n_flagged_no_cs))

n_analyzed,n_with_cs,n_fallback,n_flagged_no_cs,pct
<int>,<int>,<int>,<int>,<dbl>
16296,338,24,0,7.1


Across all 12 datasets, SER fallback was used in 7.1% (24/338) of study-region analyses with at least one credible set in the final SuSiE-rRSS result; a further 0 flagged analyses reported no credible set after fallback.


In [3]:
## ---- B4: APOE region (apoe_block in R/utils.R; Results l.337) ----
apoe <- flags %>% filter(block_id == apoe_block) %>% fallback_summary()
apoe
cat(sprintf("The APOE region remained challenging, with %.1f%% (%d/%d) of study-region analyses with a credible set assigned to SER fallback.\n",
            apoe$pct, apoe$n_fallback, apoe$n_with_cs))

## which studies fell back at APOE
flags %>% filter(block_id == apoe_block) %>% select(study, fallback, n_cs) %>% arrange(desc(fallback), study)

n_analyzed,n_with_cs,n_fallback,n_flagged_no_cs,pct
<int>,<int>,<int>,<int>,<dbl>
12,11,9,0,81.8


The APOE region remained challenging, with 81.8% (9/11) of study-region analyses with a credible set assigned to SER fallback.


study,fallback,n_cs
<chr>,<lgl>,<int>
AD_Bellenguez_2022,TRUE,1
AD_Bellenguez_EADB_2022,TRUE,1
AD_Bellenguez_main_2026,TRUE,1
AD_Bellenguez_nobiobank_2026,TRUE,1
AD_Bellenguez_noproxy_2026,TRUE,1
AD_Kunkle_Stage1_2019,TRUE,1
AD_Wightman_Excluding23andMe_2021,TRUE,1
AD_Wightman_ExcludingUKBand23andME_2021,TRUE,1
AD_Wightman_Full_2021,TRUE,1


In [4]:
## ---- B5: per-dataset fallback frequency (Fig S11a; denominator = analyses with a final credible set) ----
per_study <- flags %>%
  group_by(study) %>%
  fallback_summary() %>%
  mutate(fallback_percent = if_else(n_with_cs > 0, 100 * n_fallback / n_with_cs, 0),
         study_label      = unname(cohort_labels[study])) %>%
  select(-pct) %>%
  arrange(desc(fallback_percent))
stopifnot(!any(is.na(per_study$study_label)), sum(per_study$n_with_cs) == nrow(cs_per_region))
per_study

save_data(per_study, data_name)

study,n_analyzed,n_with_cs,n_fallback,n_flagged_no_cs,fallback_percent,study_label
<chr>,<int>,<int>,<int>,<int>,<dbl>,<chr>
AD_Wightman_Full_2021,1358,26,11,0,42.307692,Wight. full
AD_Wightman_Excluding23andMe_2021,1358,21,4,0,19.047619,Wight. -23
AD_Bellenguez_EADB_2022,1358,13,2,0,15.384615,Bell. EADB
AD_Wightman_ExcludingUKBand23andME_2021,1358,19,2,0,10.526316,Wight. -UKB/23
AD_Kunkle_Stage1_2019,1358,10,1,0,10.000000,Kunkle
AD_Bellenguez_2022,1358,46,1,0,2.173913,Bell.
AD_Bellenguez_nobiobank_2026,1358,49,1,0,2.040816,EADB. 2026 -biobank
AD_Bellenguez_noproxy_2026,1358,63,1,0,1.587302,EADB. 2026 -proxy
AD_Bellenguez_main_2026,1358,75,1,0,1.333333,EADB. 2026


In [5]:
## ---- B6: distinct regions with multiple SuSiE-rRSS credible sets (Results l.322-323) ----
## credible sets per region from the eb_mix export (cs_per_region, cell 1)
multi <- cs_per_region %>% filter(n_cs > 1)

b6 <- tibble(n_with_cs          = nrow(cs_per_region),          # text: 338
             n_multi            = nrow(multi),                  # text: 56
             pct_multi          = round(100 * n_multi / n_with_cs, 1),
             n_distinct_regions = n_distinct(multi$block_id))
b6
cat(sprintf("SuSiE-rRSS reported multiple credible sets in %.1f%% (%d/%d) of study-region analyses with at least one reported set, spanning %d distinct genomic regions across the 12 GWAS datasets.\n",
            b6$pct_multi, b6$n_multi, b6$n_with_cs, b6$n_distinct_regions))

## the regions, with how many datasets report multiple sets there
multi %>% count(block_id, name = "n_datasets") %>% arrange(desc(n_datasets), block_id)

n_with_cs,n_multi,pct_multi,n_distinct_regions
<int>,<int>,<dbl>,<int>
338,56,16.6,17


SuSiE-rRSS reported multiple credible sets in 16.6% (56/338) of study-region analyses with at least one reported set, spanning 17 distinct genomic regions across the 12 GWAS datasets.


block_id,n_datasets
<chr>,<int>
chr19_90949_1507536,8
chr8_26225312_29469590,8
chr11_119829118_122612025,7
chr16_81293081_82644764,4
chr19_53788025_55215821,4
chr14_92293029_94303139,3
chr15_61125463_66680537,3
chr2_105444433_106385118,3
chr2_125689597_127728648,3
